Search 이외 임베딩 활용 방법을 알려 드립니다
- ABC news topic modeling
    - Clustering
    - 정보의 다양성 측정
    - Outlier detection

    
=> VectorDB에 저장하고자 하는 컨텐츠에 대한 검수 및 전처리

---

In [2]:
import pandas as pd
import os
import json
import openai
from openai import OpenAI
import numpy as np
from tqdm.notebook import tqdm, trange
from sklearn.cluster import KMeans
from utils import create_embeddings

# initialize openai
from dotenv import load_dotenv
load_dotenv()
openai.api_key = os.environ["OPENAI_API_KEY"]

# How To (ABC News)

## 1. Clustering
- 2020년에 어떤 주제들의 뉴스들이 있었을까?
##### => __각 문서의 주제 탐색 / 유사 문서 그룹핑__

In [4]:
df = pd.read_csv("dataset/abcnews_2020.csv")

(비용 발생 주의) batch 별로 embedding화

In [5]:
batch_size = 2000
headline_emb = list()

headline = df['headline_text'].tolist()

for i in trange(0, len(headline), batch_size):
    i_end = min(len(headline), i+batch_size)
    data_batch = headline[i:i_end]

    tmp_emb = create_embeddings(data_batch)
    headline_emb.extend(tmp_emb)

  0%|          | 0/2 [00:00<?, ?it/s]

In [6]:
df['headline_emb'] = headline_emb

In [8]:
df

,publish_date,headline_text,headline_emb
0,20200101,a new type of resolution for the new year,"[-0.0299206729978323, 0.02757796086370945, 0.0..."
1,20200101,adelaide records driest year in more than a de...,"[0.02336890995502472, 0.02421138435602188, 0.0..."
2,20200101,adelaide riverbank catches alight after new ye...,"[0.008502775803208351, -0.006753506604582071, ..."
3,20200101,adelaides 9pm fireworks spark blaze on riverbank,"[0.03186402469873428, 3.975793561039609e-07, 0..."
4,20200101,archaic legislation governing nt women propert...,"[0.05418633669614792, 0.061872921884059906, 0...."
...,...,...,...
2442,20200131,who coronavirus global emergency,"[-0.025181135162711143, -0.0046852827072143555..."
2443,20200131,who declares coronavirus outbreak as global he...,"[-0.048844143748283386, -0.03386720269918442, ..."
2444,20200131,will travel insurance cover trip cancelled ove...,"[-0.017333682626485825, -0.020523475483059883,..."
2445,20200131,world youngest leader 33 years old offers hope...,"[0.02678145468235016, -0.034070923924446106, 0..."


In [ ]:
df.head()

,publish_date,headline_text,headline_emb
0,20200101,a new type of resolution for the new year,"[-0.0299206729978323, 0.02757796086370945, 0.0..."
1,20200101,adelaide records driest year in more than a de...,"[0.02336890995502472, 0.02421138435602188, 0.0..."
2,20200101,adelaide riverbank catches alight after new ye...,"[0.008502775803208351, -0.006753506604582071, ..."
3,20200101,adelaides 9pm fireworks spark blaze on riverbank,"[0.03186402469873428, 3.975793561039609e-07, 0..."
4,20200101,archaic legislation governing nt women propert...,"[0.05418633669614792, 0.061872921884059906, 0...."


In [ ]:
# df.to_csv("abcnews_2020_emb.csv", index=False)

k-means를 활용하여 주요 토픽별 cluster 생성

<img src="https://static.javatpoint.com/tutorial/machine-learning/images/k-means-clustering-algorithm-in-machine-learning.png" width="500" height="300"/>
<br>
출처 : https://static.javatpoint.com/tutorial/machine-learning/images/k-means-clustering-algorithm-in-machine-learning.png

In [ ]:
# df = pd.read_csv("abcnews_2020_emb.csv")

In [ ]:
# type(df.loc[0, 'headline_emb'])

# JSON 문자열 -> 리스트 변환 ( 직접 임베딩 처리한 경우, 필요없음 )
# df['headline_emb'] = df['headline_emb'].apply(json.loads)

# type(df.loc[0, 'headline_emb'])

In [14]:
df.head()

,publish_date,headline_text,headline_emb
0,20200101,a new type of resolution for the new year,"[-0.0299206729978323, 0.02757796086370945, 0.0..."
1,20200101,adelaide records driest year in more than a de...,"[0.02336890995502472, 0.02421138435602188, 0.0..."
2,20200101,adelaide riverbank catches alight after new ye...,"[0.008502775803208351, -0.006753506604582071, ..."
3,20200101,adelaides 9pm fireworks spark blaze on riverbank,"[0.03186402469873428, 3.975793561039609e-07, 0..."
4,20200101,archaic legislation governing nt women propert...,"[0.05418633669614792, 0.061872921884059906, 0...."


In [19]:
df.head(2)

,publish_date,headline_text,headline_emb
0,20200101,a new type of resolution for the new year,"[-0.0299206729978323, 0.02757796086370945, 0.0..."
1,20200101,adelaide records driest year in more than a de...,"[0.02336890995502472, 0.02421138435602188, 0.0..."


In [20]:
clusters = KMeans(n_clusters=15, random_state=0).fit_predict(df['headline_emb'].tolist())
df['cluster'] = clusters

In [21]:
df.head(2)

,publish_date,headline_text,headline_emb,cluster
0,20200101,a new type of resolution for the new year,"[-0.0299206729978323, 0.02757796086370945, 0.0...",10
1,20200101,adelaide records driest year in more than a de...,"[0.02336890995502472, 0.02421138435602188, 0.0...",6


In [22]:
df.loc[df['cluster']==1]

,publish_date,headline_text,headline_emb,cluster
162,20200103,nick kyrgios kicks off australias atp cup chal...,"[-0.055879849940538406, 0.018472496420145035, ...",1
199,20200104,australia marnus labuschagne stars vs new zeal...,"[-0.018468916416168213, 0.018722830340266228, ...",1
201,20200104,bushfire help sparked by ashleigh barty pink a...,"[-0.0062495265156030655, -0.032479289919137955...",1
249,20200104,wrong anthem played for moldova at atp cup,"[-0.03379949927330017, 0.012539973482489586, 0...",1
297,20200105,sasha zhoya turns back on australia athletics ...,"[0.026829250156879425, 0.0323784202337265, 0.0...",1
...,...,...,...,...
2314,20200130,rafael nadal agitated by chair umpire after gi...,"[-0.05127469450235367, 0.030256222933530807, 0...",1
2315,20200130,rafael nadal loses to dominic thiem australian...,"[-0.018225979059934616, 0.014112891629338264, ...",1
2354,20200131,australian open dominic thiem beats alexander ...,"[-0.01040242612361908, 0.03772582858800888, 0....",1
2355,20200131,australian open has delivered more that we cou...,"[-0.018946589902043343, 0.051692429929971695, ...",1


In [30]:
df.loc[df['cluster']==13]

,publish_date,headline_text,headline_emb,cluster
62,20200102,angus taylor investigation referred to afp,"[-0.06788653135299683, 0.019422084093093872, 0...",13
63,20200102,anthony albanese continues to call for action,"[-0.018651524558663368, 0.0026623220182955265,...",13
88,20200102,ghan plans revealed in john howard government ...,"[-0.05026073381304741, 0.001838597236201167, 0...",13
90,20200102,group of cobargo residents vent anger at pm yo...,"[0.020611368119716644, 0.014940742403268814, 0...",13
112,20200102,scott morrison responds to unwelcome reception...,"[-0.022306786850094795, 0.013650421053171158, ...",13
...,...,...,...,...
2299,20200130,laura tingle on the sports grants scandal,"[-0.007683703675866127, -0.019283480942249298,...",13
2301,20200130,liberal mp returns to parliament despite inves...,"[0.014248023740947247, -0.006134026683866978, ...",13
2365,20200131,coronavirus adds to scott morrisons many polit...,"[-0.022894103080034256, -0.017467066645622253,...",13
2404,20200131,nsw balranald council placed under administrat...,"[-0.03615427762269974, 0.05161817744374275, 0....",13


## 2. 정보의 다양성 (Diversity) 측정

- 각 클러스터 내에 있는 뉴스들은 얼마나 유사한 정보를 담고 있을까?

In [51]:
from sklearn.metrics.pairwise import cosine_similarity

def calculate_diversity(df, column_name):
    """
    Calculates the diversity of a set of embeddings based on cosine distance.
    
    :param embeddings: NumPy array of embeddings
    :return: The average cosine distance between embeddings, higher means more diverse
    """
    # 각각의 임베딩끼리 모두 pairwise cosine similarity를 계산
    embeddings = np.vstack(df[column_name])  #  2D NumPy 배열 변환 ( 데이터 자체는 동일 )
    cosine_sim = cosine_similarity(embeddings)
    
    # self-comparisons (diagonal elements)를 제외하고 cosine similarity 계산
    np.fill_diagonal(cosine_sim, np.nan) # 본인과의 similarity는 제외
    avg_distance = np.nanmean(cosine_sim)
    
    return cosine_sim, avg_distance


In [45]:
df['headline_emb']

0       [-0.0299206729978323, 0.02757796086370945, 0.0...
1       [0.02336890995502472, 0.02421138435602188, 0.0...
2       [0.008502775803208351, -0.006753506604582071, ...
3       [0.03186402469873428, 3.975793561039609e-07, 0...
4       [0.05418633669614792, 0.061872921884059906, 0....
                              ...                        
2442    [-0.025181135162711143, -0.0046852827072143555...
2443    [-0.048844143748283386, -0.03386720269918442, ...
2444    [-0.017333682626485825, -0.020523475483059883,...
2445    [0.02678145468235016, -0.034070923924446106, 0...
2446    [-0.028029361739754677, 0.019691335037350655, ...
Name: headline_emb, Length: 2447, dtype: object

In [57]:
dist, avg = calculate_diversity(df, 'headline_emb')

In [43]:
avg

np.float64(0.19837453021458046)

In [35]:
diversity_score = {k:calculate_diversity(df.loc[df['cluster']==k], 'headline_emb')[1] for k in range(0, 15)}

In [36]:
diversity_score

{0: np.float64(0.29813683619055376),
 1: np.float64(0.426947920226263),
 2: np.float64(0.2861774929040097),
 3: np.float64(0.19257912691351223),
 4: np.float64(0.2580676409291806),
 5: np.float64(0.8579312130529011),
 6: np.float64(0.3707788550176585),
 7: np.float64(0.4505822347026101),
 8: np.float64(0.36340071729512247),
 9: np.float64(0.3726228793938779),
 10: np.float64(0.15389151951295643),
 11: np.float64(0.3722109819690211),
 12: np.float64(0.42575742857409643),
 13: np.float64(0.3110738742241808),
 14: np.float64(0.19758120468676152)}

In [37]:
df.loc[df['cluster']==7]

,publish_date,headline_text,headline_emb,cluster
3,20200101,adelaides 9pm fireworks spark blaze on riverbank,"[0.03186402469873428, 3.975793561039609e-07, 0...",7
7,20200101,bushfire relief: how you can help frontline se...,"[-0.006802694872021675, -0.018417952582240105,...",7
8,20200101,bushfires what now for stranded,"[0.004284844268113375, 0.009887057356536388, 0...",7
14,20200101,family defends home from fire at goongerah,"[0.013996439054608345, 0.025780055671930313, 0...",7
21,20200101,karen lissa describes the bushfire swept throu...,"[-0.010012606158852577, -0.020255541428923607,...",7
...,...,...,...,...
2342,20200130,weather event making australian bushfires wors...,"[-0.027965623885393143, 0.042904291301965714, ...",7
2350,20200131,act enters state of emergency amid bushfire th...,"[-0.01568206027150154, 0.011571952141821384, 0...",7
2360,20200131,brumbies game threatened by smoke and heat,"[-0.0339898057281971, 0.02021198906004429, 0.0...",7
2430,20200131,tourism industry questions nsw response to bus...,"[-0.018678603693842888, -0.024815283715724945,...",7


## 4. Outlier detection
- 각 클러스터 내에 속하지 않는 정보들이 있을까?

<img src="https://miro.medium.com/v2/resize:fit:725/1*y3wXEId0poYUIzCD3HBh4w.png"/>
<br>
출처 : https://miro.medium.com/v2/resize:fit:725/1*y3wXEId0poYUIzCD3HBh4w.png

In [38]:
from sklearn.ensemble import IsolationForest

In [39]:
cluster = df.loc[df['cluster']==10]

In [ ]:
iso_forest = IsolationForest(contamination=0.05)  # Adjust contamination as needed
anomalies = iso_forest.fit_predict(cluster['headline_emb'].tolist())

anomalous_headlines = np.array(cluster['headline_text'].tolist())[anomalies == -1]

In [59]:
print("Anomalous Headlines:", anomalous_headlines)

Anomalous Headlines: ['dozens of residents are queuing to get into' 'nols newpm 0401'
 'think twice before you share cute photos of animals'
 'baby yoda name still secret gender probably male the mandalorian'
 'a pocket guide to climate change' 'national finals campdraft cancelled'
 'image template' 'beef breeders to uruguay bed and breakfast'
 'starting high school dr kaylene henderson'
 'contaminated wintulichs mettwurst recalled'
 'from bats to snakes to people: whats the source of'
 'starting costs graduate teachers in the classroom'
 'star trek will boldly go on and on and on'
 'david whiting on the red mass tradition']


In [41]:
anomalies

array([ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, -1,  1,  1,  1,  1,  1,
        1,  1,  1,  1,  1,  1,  1,  1,  1, -1,  1,  1,  1,  1,  1,  1,  1,
        1,  1, -1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, -1,  1,  1,  1,
        1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
        1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, -1,  1,  1,
        1,  1,  1,  1,  1, -1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
        1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, -1,  1,  1,
        1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
        1, -1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
        1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
        1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
        1,  1, -1,  1,  1, -1,  1,  1, -1,  1,  1,  1,  1,  1, -1,  1,  1,
        1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, -1,  1,  1,  1,
        1,  1,  1,  1,  1

In [42]:
anomalous_headlines

array(['dozens of residents are queuing to get into', 'nols newpm 0401',
       'think twice before you share cute photos of animals',
       'baby yoda name still secret gender probably male the mandalorian',
       'a pocket guide to climate change',
       'national finals campdraft cancelled', 'image template',
       'beef breeders to uruguay bed and breakfast',
       'starting high school dr kaylene henderson',
       'contaminated wintulichs mettwurst recalled',
       'from bats to snakes to people: whats the source of',
       'starting costs graduate teachers in the classroom',
       'star trek will boldly go on and on and on',
       'david whiting on the red mass tradition'], dtype='<U64')

단순히 텍스트를 embedding화 하는 것에서 더 나아가, <br>
텍스트를 특징별로 묶거나 유관하지 않다고 판단되는 텍스트는 제외하는 등, 컨텐츠 자체를 preprocessing/탐색 하는데에 활용 가능

--END--